In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. IMU ATTITUDE, DRIFT, NOISE & CALIBRATION
# ==========================================
CALIBRATION_MODES = {
    "Uncalibrated (Raw Gyro)": {
        "remove_bias": False,
        "use_accel_fusion": False,
        "alpha": 1.0,
        "desc": "Raw gyroscope integration without bias calibration; uncalibrated bias accumulates over time and causes drift."
    },
    "Bias Calibrated (Gyro Only)": {
        "remove_bias": True,
        "use_accel_fusion": False,
        "alpha": 1.0,
        "desc": "Removes estimated gyro bias prior to integration, reducing drift while keeping high-frequency noise."
    },
    "Calibrated + Complementary Filter": {
        "remove_bias": True,
        "use_accel_fusion": True,
        "alpha": 0.96,
        "desc": "Combines bias-calibrated gyro integration with accelerometer tilt correction to suppress drift and balance noise."
    },
}

def simulate_imu_attitude(roll_deg, pitch_deg, drift_rate_dps, noise_level, cal_mode, duration_s=20.0, dt=0.02, seed=202):
    cfg = CALIBRATION_MODES[cal_mode]
    n = int(duration_s / dt) + 1
    t = np.linspace(0.0, duration_s, n)
    u = t / max(duration_s, 1e-6)

    # Smooth true roll and pitch profile [deg]
    roll_true = roll_deg * np.sin(np.pi * u) * (1.0 - 0.15 * np.cos(2.0 * np.pi * u))
    pitch_true = pitch_deg * np.sin(0.5 * np.pi * u)

    roll_rate_true = np.gradient(roll_true, dt)
    pitch_rate_true = np.gradient(pitch_true, dt)

    rng = np.random.default_rng(seed)
    gyro_noise_dps = rng.normal(0.0, 0.85 * max(noise_level, 0.0), size=n)
    accel_roll_meas = roll_true + rng.normal(0.0, 1.40 * max(noise_level, 0.0), size=n)

    # True bias error [deg/s]
    true_bias_dps = drift_rate_dps
    gyro_meas_dps = roll_rate_true + true_bias_dps + gyro_noise_dps

    # Calibration estimate (small residual bias ~8% if calibrated)
    est_bias_dps = 0.08 * true_bias_dps if cfg["remove_bias"] else 0.0
    rate_clean_dps = gyro_meas_dps - est_bias_dps

    roll_est = np.zeros(n)
    roll_est[0] = roll_true[0]

    for i in range(1, n):
        gyro_prop = roll_est[i - 1] + rate_clean_dps[i] * dt
        if cfg["use_accel_fusion"]:
            roll_est[i] = cfg["alpha"] * gyro_prop + (1.0 - cfg["alpha"]) * accel_roll_meas[i]
        else:
            roll_est[i] = gyro_prop

    # Pitch estimate (follows true pitch with small calibration/noise offset)
    pitch_est = pitch_true + (0.45 * drift_rate_dps * u if not cfg["remove_bias"] else 0.05 * drift_rate_dps * u) + rng.normal(0.0, 0.4 * noise_level, size=n)

    err = roll_est - roll_true
    rmse = float(np.sqrt(np.mean(err**2)))
    final_err = float(err[-1])
    final_pitch = float(pitch_est[-1])
    final_roll_est = float(roll_est[-1])

    return pd.DataFrame({
        "time_s": t,
        "roll_true_deg": roll_true,
        "roll_est_deg": roll_est,
        "pitch_est_deg": pitch_est,
    }), {
        "rmse": rmse,
        "final_err": final_err,
        "final_roll_est": final_roll_est,
        "final_pitch": final_pitch,
    }

def build_imu_why_explanation(cal_mode, drift_dps, noise_level, stats):
    reasons = []

    if cal_mode == "Uncalibrated (Raw Gyro)" and abs(drift_dps) > 0.2:
        reasons.append(f"uncalibrated gyro drift ({drift_dps:+.2f} °/s) integrates over time, causing a {stats['final_err']:+.1f}° final roll error")
    elif cal_mode == "Bias Calibrated (Gyro Only)":
        reasons.append(f"bias calibration removes most of the {drift_dps:+.2f} °/s offset, while gyro-only integration still leaves {stats['final_err']:+.1f}° final error")
    elif cal_mode == "Calibrated + Complementary Filter":
        reasons.append(f"calibration plus complementary filtering anchors roll against accelerometer tilt, keeping final error at {stats['final_err']:+.1f}°")
    else:
        reasons.append(f"low drift ({drift_dps:+.2f} °/s) keeps final error modest at {stats['final_err']:+.1f}°")

    if noise_level >= 1.2:
        reasons.append(f"high IMU noise ({noise_level:.1f}) increases short-term jitter and brings RMSE to {stats['rmse']:.2f}°")
    else:
        reasons.append(f"moderate sensor noise yields an overall RMSE of {stats['rmse']:.2f}°")

    return f"<b>Why This Result Happened ({cal_mode}):</b> The attitude indicator behaves this way because " + "; and ".join(reasons) + "."

# ==========================================
# 2. 2D ATTITUDE INDICATOR + TIME RESPONSE
# ==========================================
def draw_attitude_indicator(roll_deg, pitch_deg, drift_dps, noise_level, cal_mode, run_idx):
    df, stats = simulate_imu_attitude(
        roll_deg, pitch_deg, drift_dps, noise_level, cal_mode, seed=202 + run_idx
    )
    why_text = build_imu_why_explanation(cal_mode, drift_dps, noise_level, stats)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_ai, ax_ts) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.0, 1.45]})

    # --- LEFT: 2D Attitude Indicator (Artificial Horizon) ---
    # Draw circular instrument frame
    theta_c = np.linspace(0, 2 * np.pi, 200)
    ax_ai.plot(np.cos(theta_c), np.sin(theta_c), color="#0f172a", lw=2.5)

    # True horizon line tilted by true final roll and shifted by pitch
    true_roll_rad = np.deg2rad(df["roll_true_deg"].iloc[-1])
    pitch_shift = np.clip(stats["final_pitch"] / 35.0, -0.55, 0.55)
    hx = np.linspace(-0.92, 0.92, 50)
    hy_true = pitch_shift + np.tan(true_roll_rad) * hx
    mask_in = (hx**2 + hy_true**2) <= 0.96**2
    ax_ai.plot(hx[mask_in], hy_true[mask_in], color="#16a34a", lw=2.2, ls="--", label="True Horizon")

    # Estimated horizon line (affected by drift/noise/calibration)
    est_roll_rad = np.deg2rad(stats["final_roll_est"])
    hy_est = pitch_shift + np.tan(est_roll_rad) * hx
    mask_est = (hx**2 + hy_est**2) <= 0.96**2
    ax_ai.plot(hx[mask_est], hy_est[mask_est], color="#dc2626", lw=2.5, label="Estimated Horizon")

    # Fixed aircraft reference wings at center
    ax_ai.plot([-0.45, -0.15], [0.0, 0.0], color="#2563eb", lw=3.2)
    ax_ai.plot([0.15, 0.45], [0.0, 0.0], color="#2563eb", lw=3.2)
    ax_ai.scatter([0.0], [0.0], color="#2563eb", s=45, zorder=7)

    ax_ai.set_title(f"2D Attitude Indicator (Roll={stats['final_roll_est']:+.1f}°, Pitch={stats['final_pitch']:+.1f}°)", fontsize=10.5, fontweight="bold", pad=10)
    ax_ai.set_xlim(-1.12, 1.12)
    ax_ai.set_ylim(-1.12, 1.12)
    ax_ai.set_aspect("equal", adjustable="box")
    ax_ai.axis("off")
    ax_ai.legend(loc="lower center", ncol=2, fontsize=8, frameon=True, facecolor="white")

    # --- RIGHT: Roll Angle vs. Time ---
    ax_ts.plot(df["time_s"], df["roll_true_deg"], color="#16a34a", ls="--", lw=2.0, label="True Roll [deg]")
    ax_ts.plot(df["time_s"], df["roll_est_deg"], color="#ea580c", lw=2.0, label="Estimated Roll [deg]")

    ax_ts.set_title(f"Roll Tracking vs. Time — {cal_mode}", fontsize=11, fontweight="bold", pad=10)
    ax_ts.set_xlabel("Time [s]", fontsize=9.5)
    ax_ts.set_ylabel("Roll Angle [deg]", fontsize=9.5)
    ax_ts.legend(loc="upper right", fontsize=8.5, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:13px 17px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>Mode:</b> <span style="color:#7dd3fc;font-weight:700;">{cal_mode}</span> &nbsp;|&nbsp;
            <b>Roll RMSE:</b> {stats['rmse']:.2f}° &nbsp;|&nbsp;
            <b>Final Roll Error:</b> {stats['final_err']:+.2f}° &nbsp;|&nbsp;
            <b>Gyro Drift:</b> {drift_dps:+.2f} °/s
        </div>
        <div style="font-size:12.5px;color:#e2e8f0;margin-top:6px;line-height:1.55;">
            {why_text}
        </div>
        <div style="font-size:12px;color:#94a3b8;margin-top:4px;line-height:1.45;">
            {CALIBRATION_MODES[cal_mode]['desc']}
        </div>
    </div>
    """))

# ==========================================
# 3. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "155px"}
ctrl_layout = widgets.Layout(width="345px")

w_cal   = widgets.Dropdown(options=list(CALIBRATION_MODES.keys()), value="Uncalibrated (Raw Gyro)", description="Calibration Mode:", style=lbl_style, layout=ctrl_layout)
w_roll  = widgets.FloatSlider(value=20.0, min=-40.0, max=40.0, step=2.0,  description="Peak Roll [deg]:", readout_format="+.1f", style=lbl_style, layout=ctrl_layout)
w_pitch = widgets.FloatSlider(value=8.0,  min=-25.0, max=25.0, step=1.0,  description="Peak Pitch [deg]:", readout_format="+.1f", style=lbl_style, layout=ctrl_layout)
w_drift = widgets.FloatSlider(value=1.50, min=-4.00, max=4.00, step=0.10, description="Gyro Drift [°/s]:", readout_format="+.2f", style=lbl_style, layout=ctrl_layout)
w_noise = widgets.FloatSlider(value=1.0,  min=0.0,   max=3.0,  step=0.2,  description="IMU Noise:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_ai = widgets.Button(description="Run IMU Test", button_style="primary", icon="compass", layout=widgets.Layout(width="210px", height="36px"))
out_ai = widgets.Output(layout=widgets.Layout(width="100%"))
ai_cnt = {"n": 0}

def refresh_ai(*args):
    ai_cnt["n"] += 1
    with out_ai:
        clear_output(wait=True)
        draw_attitude_indicator(
            w_roll.value, w_pitch.value, w_drift.value, w_noise.value, w_cal.value, ai_cnt["n"]
        )

btn_ai.on_click(refresh_ai)
for c in [w_cal, w_roll, w_pitch, w_drift, w_noise]:
    c.observe(refresh_ai, names="value")

header_ai = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">IMU Attitude Indicator — Drift, Noise &amp; Calibration Lab</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Test roll/pitch attitude indication under gyro drift, sensor noise, and calibration filtering
        </div>
    </div>
""")

controls_ai = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_cal, w_roll, w_pitch], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_drift, w_noise, btn_ai], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_ai, controls_ai, out_ai)
refresh_ai()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))